In [0]:
#%pip install faiss-cpu sentence-transformers
#%restart_python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 142.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.3/553.3 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.0/146.0 MB 187.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 147.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 117.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 94.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.7/798.7 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 147.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 106.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 20.2 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.1.7
    Not uninstalling click at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/en

In [0]:
%sql describe detail workspace.default.sales_data_embeddings

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,99e729c9-94fc-461d-a354-ea9b83fb38bf,workspace.default.sales_data_embeddings,null,,2026-02-19T05:33:07.692Z,2026-02-19T13:18:40.000Z,List(),List(),1,13833,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
#%pip install deltalake
#%restart_python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.1/37.1 MB 163.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 112.4 MB/s eta 0:00:00
  Attempting uninstall: deprecated
    Found existing installation: Deprecated 1.2.13
    Not uninstalling deprecated at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-cd6b5ddd-cc1b-4965-bd93-0ee9ed9bca19
    Can't uninstall 'Deprecated'. No files were found to uninstall.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import faiss
import numpy as np
import pandas as pd
from delta.tables import DeltaTable
 

table_name = "workspace.default.sales_data_embeddings"

# Load the Delta table
dt =   DeltaTable.forName(spark, table_name) 
df = dt.toDF()
 
flat_df = df.explode('embeddings')
 
display(flat_df)
 # Collect into a Python list
#data_to_index = [row.embeddings  for row in flat_df.collect()]
#display(data_to_index)


#dim=np.array(data_to_index[0]).shape[0]
#print(dim)
 

 

---------------------------------------------------------------------------
PySparkAttributeError                     Traceback (most recent call last)
File <command-7441240896825477>, line 13
     10 dt =   DeltaTable.forName(spark, table_name) 
     11 df = dt.toDF()
---> 13 flat_df = df.explode('embeddings')
     15 display(flat_df)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1770, in DataFrame.__getattr__(self, name)
   1767 # END-EDGE
   1769 if name not in self.columns:
-> 1770     raise PySparkAttributeError(
   1771         errorClass="ATTRIBUTE_NOT_SUPPORTED", messageParameters={"attr_name": name}
   1772     )
   1774 return self._col(name)

PySparkAttributeError: [ATTRIBUTE_NOT_SUPPORTED] Attribute `explode` is not supported.

In [0]:
import faiss
import numpy as np
import pandas as pd
import torch
from pyspark.sql.functions import explode
from pyspark.sql.functions import udf, concat_ws, col
 
table_name = "workspace.default.sales_data_embeddings"
df = spark.read.table(table_name)
df=df.filter(col("COUNTRY") == "Canada") 
#data_to_index=df.select('embeddings').rdd.flatMap(lambda x: x).collect() <--this does not work with serverless compute
#flat_df = df.select(explode("embeddings").alias("val"))

flat_df=df.select('embeddings').rdd.flatMap(lambda x: x).collect()

# Collect into a Python list
data_to_index = [row.val for row in flat_df.collect()]

display(data_to_index)

clean_data_to_index = [0 if x is None else np.stack(x).astype(np.float32) for x in data_to_index if x is not None]
#display(np.array(data_to_index)[0])

#dim=np.array(clean_data_to_index[0]).shape[0]
#print(dim)

 

_1,_2,_3,_4,_5,_6,_7,_8,_9,_10,_11,_12,_13,_14,_15,_16,_17,_18,_19,_20,_21,_22,_23,_24,_25,_26,_27,_28,_29,_30,_31,_32,_33,_34,_35,_36,_37,_38,_39,_40,_41,_42,_43,_44,_45,_46,_47,_48,_49,_50,_51,_52,_53,_54,_55,_56,_57,_58,_59,_60,_61,_62,_63,_64,_65,_66,_67,_68,_69,_70,_71,_72,_73,_74,_75,_76,_77,_78,_79,_80,_81,_82,_83,_84,_85,_86,_87,_88,_89,_90,_91,_92,_93,_94,_95,_96,_97,_98,_99,_100,_101,_102,_103,_104,_105,_106,_107,_108,_109,_110,_111,_112,_113,_114,_115,_116,_117,_118,_119,_120,_121,_122,_123,_124,_125,_126,_127,_128,_129,_130,_131,_132,_133,_134,_135,_136,_137,_138,_139,_140,_141,_142,_143,_144,_145,_146,_147,_148,_149,_150,_151,_152,_153,_154,_155,_156,_157,_158,_159,_160,_161,_162,_163,_164,_165,_166,_167,_168,_169,_170,_171,_172,_173,_174,_175,_176,_177,_178,_179,_180,_181,_182,_183,_184,_185,_186,_187,_188,_189,_190,_191,_192,_193,_194,_195,_196,_197,_198,_199,_200,_201,_202,_203,_204,_205,_206,_207,_208,_209,_210,_211,_212,_213,_214,_215,_216,_217,_218,_219,_220,_221,_222,_223,_224,_225,_226,_227,_228,_229,_230,_231,_232,_233,_234,_235,_236,_237,_238,_239,_240,_241,_242,_243,_244,_245,_246,_247,_248,_249,_250,_251,_252,_253,_254,_255,_256,_257,_258,_259,_260,_261,_262,_263,_264,_265,_266,_267,_268,_269,_270,_271,_272,_273,_274,_275,_276,_277,_278,_279,_280,_281,_282,_283,_284,_285,_286,_287,_288,_289,_290,_291,_292,_293,_294,_295,_296,_297,_298,_299,_300,_301,_302,_303,_304,_305,_306,_307,_308,_309,_310,_311,_312,_313,_314,_315,_316,_317,_318,_319,_320,_321,_322,_323,_324,_325,_326,_327,_328,_329,_330,_331,_332,_333,_334,_335,_336,_337,_338,_339,_340,_341,_342,_343,_344,_345,_346,_347,_348,_349,_350,_351,_352,_353,_354,_355,_356,_357,_358,_359,_360,_361,_362,_363,_364,_365,_366,_367,_368,_369,_370,_371,_372,_373,_374,_375,_376,_377,_378,_379,_380,_381,_382,_383,_384
-0.04689415171742439,0.008603306487202644,-0.02265937440097332,0.028424933552742004,-0.016441620886325836,0.04323834925889969,-0.003354213899001479,0.04600662365555763,0.01874914951622486,-0.006305520422756672,-0.04859526455402374,-0.05189001187682152,-0.021727677434682846,-0.04514913633465767,-0.08174975961446762,-0.03794606402516365,0.0018222762737423182,-0.03130447492003441,0.03655453771352768,-0.006068650633096695,-0.0357791893184185,0.0031921742483973503,0.02369670942425728,0.02579049952328205,0.06760584563016891,0.008042027242481709,-0.042494382709264755,-0.017651395872235298,0.03929324820637703,-0.026949521154165268,-0.03952465206384659,0.052139170467853546,0.005269752815365791,0.04609372094273567,0.09727352857589722,0.008921344764530659,-0.02803083136677742,-0.0014230695087462664,0.04780790954828262,-0.034205470234155655,0.029886087402701378,0.04473525285720825,-0.025775399059057236,0.05783117190003395,-0.03354726359248161,0.025769459083676338,-5.058501265011728E-4,-3.8295875128824264E-5,0.0043524643406271935,0.028478819876909256,0.043143611401319504,0.09515408426523209,-0.028442101553082466,-0.06286199390888214,0.1255413293838501,-0.005135979503393173,-0.09334369003772736,-0.0432264544069767,0.04252210631966591,0.016391964629292488,0.05727129057049751,-0.025202056393027306,-0.01078771147876978,-0.08116312325000763,-7.589191664010286E-4,0.014097106643021107,-0.06409107148647308,-0.005255909636616707,-0.023002684116363525,-0.05299094319343567,0.007849141024053097,0.02836630493402481,-0.01033695600926876,0.07380348443984985,-0.059106115251779556,-0.024953437969088554,0.06932315975427628,-0.08131406456232071,-0.06657025963068008,0.061759572476148605,-0.09366980940103531,-0.008616484701633453,-0.048907458782196045,0.03379686549305916,0.05032733827829361,-0.1068737581372261,0.08149705827236176,0.0586206279695034,0.021996067836880684,0.032174043357372284,0.04088409245014191,0.02492956817150116,0.04019197076559067,-0.007488275412470102,-0.08908267319202423,-0.017508000135421753,0.06405223906040192,0.07821814715862274,0.07042423635721207,-0.008071905001997948,0.07677207142114639,0.07351010292768478,0.06663424521684647,0.05009699612855911,-0.081459015

In [0]:
index=faiss.IndexFlatL2(dim)

index.add(np.array(clean_data_to_index))

display(index)

<faiss.swigfaiss_sve.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0xff8fe51d7120> >

In [0]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
query = "What is the average sales for the product with the highest sales?"
query_vector = model.encode(query)
#check  dim diemention and query_vector dimention . it should match
print('dim' + str(dim))
print('query_vector' + str(query_vector.shape))

k = 2
distance, pos = index.search(np.array([query_vector]), k)    
print('pos' + str(pos))
print('distance' + str(distance))
 
 
 

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


dim384
query_vector(384,)
pos[[0 1]]
distance[[1.5607452 1.5607452]]


In [0]:
#result=df.select('combined_text').rdd.flatMap(lambda x: x).collect()[int(pos[0][0])] <--this does not wok with severless compute
idx = int(pos[0][0])
# Collect only the 'combined_text' column as a list of Rows
# This is much faster than RDD flatMap
rows = df.select('combined_text').collect()

# Extract the text from the specific Row object
result = rows[idx]['combined_text']
display(result)

'Canadian Gift Exchange Network makes 9064.89 in 2003 at Vancouver BC in Canada Classic Cars is S10_1949 in NA deal size was Large'